In [ ]:
# Environment setup
%load_ext autoreload
%autoreload 2

import os
os.chdir("..")

In [52]:
import numpy as np
import pandas as pd

from aether.clause.nodes.nodes import SMA
from aether.clause.nodes.nodes import ADD
from aether.clause.nodes.nodes import SHIFT
from aether.clause.nodes.nodes import DIFF
from aether.clause.nodes.nodes import PctChange
from aether.clause.nodes.nodes import ShiftSign
from aether.clause.nodes.nodes import ABS
from aether.clause.nodes.nodes import DIV
from aether.clause.nodes.nodes import SUB
from aether.clause.nodes.nodes import Comparison
from aether.clause.nodes.nodes import NewHigh
from aether.clause.nodes.nodes import NewLow
from aether.clause.nodes.nodes import ZSCORE
from aether.clause.nodes.nodes import QuantileRank
from aether.clause.nodes.nodes import CORR
from aether.clause.nodes.nodes import STD
from aether.clause.nodes.nodes import SKEW
from aether.clause.nodes.nodes import DATA
from aether.clause.nodes.nodes import LargerThan
from aether.clause.nodes.nodes import SmallerThan

from aether.clause.tree import ClauseTree
from aether.clause.tree.generator import ClauseGenerator
from aether.clause.graph import ClauseGraph
from aether.clause.graph.edge import simple_edge_weight_calculator

In [5]:
# 1. Prepare Basic Nodes and ClauseTree Generator
nodes = [
    # Basic operations
    ADD(), DIV(), SUB(), ABS(), ShiftSign(),
    
    # Time series analysis
    SMA(period=5), SMA(period=10), SMA(period=20),
    SHIFT(period=1), SHIFT(period=5),
    DIFF(period=1), DIFF(period=5),
    PctChange(period=1), PctChange(period=5),
    STD(period=10), STD(period=20),
    CORR(period=10), CORR(period=20),
    
    # Data nodes
    DATA("OPEN", "BTCUSDT"),
    DATA("HIGH", "BTCUSDT"), 
    DATA("LOW", "BTCUSDT"),
    DATA("CLOSE", "BTCUSDT"),
    DATA("VOLUME", "BTCUSDT"),
]

# ClauseTree generator
generator = ClauseGenerator(nodes)

print(f"Total {len(nodes)} node types prepared.")


Loading dataframes ...: 100%|██████████| 80/80 [00:00<00:00, 119.52it/s]

Total 23 node types prepared.


In [80]:
# 2. Generate Multiple ClauseTrees
trees = []
n_trees = 8

print(f"Generating {n_trees} ClauseTrees...")

for i in range(n_trees):
    tree = generator.generate(max_depth=3)
    tree.name = f"Tree_{i}"

    if tree.iscompleted and tree.depth == 3:
        trees.append(tree)
        print(f"Tree {i}: depth={tree.depth}, nodes={len(tree.nodes)}, completed={tree.iscompleted}")

Generating 8 ClauseTrees...
Tree 1: depth=3, nodes=7, completed=True
Tree 3: depth=3, nodes=4, completed=True
Tree 5: depth=3, nodes=4, completed=True
Tree 6: depth=3, nodes=4, completed=True


In [81]:
clause_graph = ClauseGraph(name = "Graph")
clause_graph

ClauseGraph(0 nodes)

In [82]:
node_id1 = clause_graph.add_clause_tree(trees[0])
node_id2 = clause_graph.add_clause_tree(trees[1])
node_id3 = clause_graph.add_clause_tree(trees[2])

2025-09-24 09:14:32 - aether.clause.graph.base - INFO - Added ClauseTree as node id: 9e53a36f
2025-09-24 09:14:32 - aether.clause.graph.base - INFO - Added ClauseTree as node id: b8a47b8a
2025-09-24 09:14:32 - aether.clause.graph.base - INFO - Added ClauseTree as node id: 207269bd


In [70]:
clause_graph.graph.add_edge(node_id1, node_id2, weight=0.5)

In [79]:
clause_graph.get_edge_weight(node_id1, node_id2)

0.5

In [ ]:
simple_edge_weight_calculator(trees[0], trees[2])

0.9076923076923077